# Phase 2b : Comparaison Grid Search et fusion

In [1]:
import json
import torch
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from collections import defaultdict
import config
import time

from utils_indexation import SearchIndex
from utils_evaluation import evaluate_retrieval_gpu

/home/marion/MIRAGE_TFE/env_marion/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration et Chargement

In [2]:
model_names_t2i = ["siglip", "blip", "convnext_clip","eva_clip"]
model_names_i2t = ["siglip", "blip", "convnext_clip", "eva_clip"]
METRIC_CIBLE = 'R@1'


prefix_test = "test_"
all_model_names = list(set(model_names_t2i + model_names_i2t))

with open(config.BEST_WEIGHTS_FILE, 'r') as f:
    BEST_WEIGHTS = json.load(f)
print(f"Poids MIRAGE chargés depuis {config.BEST_WEIGHTS_FILE}")

Poids MIRAGE chargés depuis ./data/flickr30k/grid_search/best_weights.json


## Chargement des vecteurs de Test et cibles

In [3]:
print("\nChargement des vecteurs de Test...")
txt_vecs_test, img_vecs_test = {}, {}

for name in all_model_names:
    img_vecs_test[name] = np.load(f"{config.INDEX_DIR}/{prefix_test}{name}_img_vecs.npy")
    txt_vecs_test[name] = np.load(f"{config.INDEX_DIR}/{prefix_test}{name}_txt_vecs.npy")

modele_ref = all_model_names[0]
idx_img = SearchIndex(0); idx_img.load_from_disk(f"{config.INDEX_DIR}/{prefix_test}{modele_ref}_img")
test_img_ids = idx_img.image_ids

idx_txt = SearchIndex(0); idx_txt.load_from_disk(f"{config.INDEX_DIR}/{prefix_test}{modele_ref}_txt")
test_txt_to_img_id = idx_txt.image_ids

test_img_id_to_idx = {img_id: idx for idx, img_id in enumerate(test_img_ids)}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Cibles T2I ---
targets_t2i_test = np.array([test_img_id_to_idx[iid] for iid in test_txt_to_img_id])
targets_t2i_gpu = torch.tensor(targets_t2i_test.reshape(-1, 1), device=device)

# --- Cibles I2T ---
test_img_to_txt_indices = defaultdict(list)
for txt_idx, iid in enumerate(test_txt_to_img_id):
    test_img_to_txt_indices[test_img_id_to_idx[iid]].append(txt_idx)
targets_i2t_test = [test_img_to_txt_indices[i] for i in range(len(test_img_id_to_idx))]

max_len_i2t = max(len(t) for t in targets_i2t_test)
targets_i2t_padded = [t + [-1] * (max_len_i2t - len(t)) for t in targets_i2t_test]
targets_i2t_gpu = torch.tensor(targets_i2t_padded, device=device)


Chargement des vecteurs de Test...
  Chargé : ./data/flickr30k/index_sauvegardes/test_eva_clip_img_index.bin (1000 vecteurs)
  Chargé : ./data/flickr30k/index_sauvegardes/test_eva_clip_txt_index.bin (5000 vecteurs)


## Définition des Fonctions Génériques

In [4]:
def run_ablation_study(query_vecs, gallery_vecs, model_names, targets_gpu, best_weights, device):
    """Calcule toutes les méthodes de fusion de manière agnostique et chronomètre l'exécution."""
    results = {}
    
    # --- 1. Modèles individuels ---
    for name in model_names:
        t0 = time.time()
        S = torch.tensor(query_vecs[name], device=device) @ torch.tensor(gallery_vecs[name], device=device).T
        res = evaluate_retrieval_gpu(S, targets_gpu)
        res["Temps (s)"] = time.time() - t0
        results[f"{name} seul"] = res

    # --- 2. Addition Simple ---
    t0 = time.time()
    S_add = sum((1.0/len(model_names)) * (torch.tensor(query_vecs[n], device=device) @ torch.tensor(gallery_vecs[n], device=device).T) for n in model_names)
    res = evaluate_retrieval_gpu(S_add, targets_gpu)
    res["Temps (s)"] = time.time() - t0
    results["Fusion : Addition Simple"] = res

    # --- 3. Multiplication ---
    t0 = time.time()
    S_mult = None
    for name in model_names:
        S = torch.tensor(query_vecs[name], device=device) @ torch.tensor(gallery_vecs[name], device=device).T
        S_norm = (S - S.min()) / (S.max() - S.min() + 1e-8)
        S_mult = S_norm if S_mult is None else S_mult * S_norm
    res = evaluate_retrieval_gpu(S_mult, targets_gpu)
    res["Temps (s)"] = time.time() - t0
    results["Fusion : Multiplication"] = res

    # --- 4. Concaténation ---
    t0 = time.time()
    q_concat = np.concatenate([query_vecs[n] for n in model_names], axis=1)
    g_concat = np.concatenate([gallery_vecs[n] for n in model_names], axis=1)
    q_concat = q_concat / np.linalg.norm(q_concat, axis=1, keepdims=True)
    g_concat = g_concat / np.linalg.norm(g_concat, axis=1, keepdims=True)
    S_concat = torch.tensor(q_concat, device=device) @ torch.tensor(g_concat, device=device).T
    res = evaluate_retrieval_gpu(S_concat, targets_gpu)
    res["Temps (s)"] = time.time() - t0
    results["Fusion : Concaténation"] = res

    # --- 5. Maximum (Max Pooling) ---
    t0 = time.time()
    S_max = None
    for name in model_names:
        S = torch.tensor(query_vecs[name], device=device) @ torch.tensor(gallery_vecs[name], device=device).T
        S_norm = (S - S.min()) / (S.max() - S.min() + 1e-8)
        S_max = S_norm if S_max is None else torch.maximum(S_max, S_norm)
    res = evaluate_retrieval_gpu(S_max, targets_gpu)
    res["Temps (s)"] = time.time() - t0
    results["Fusion : Maximum"] = res

    # --- 6. RRF (Reciprocal Rank Fusion) ---
    t0 = time.time()
    k_rrf = 60 
    N_queries, N_gallery = query_vecs[model_names[0]].shape[0], gallery_vecs[model_names[0]].shape[0]
    S_rrf = torch.zeros((N_queries, N_gallery), device=device)
    for name in model_names:
        S = torch.tensor(query_vecs[name], device=device) @ torch.tensor(gallery_vecs[name], device=device).T
        sorted_indices = torch.argsort(S, dim=1, descending=True)
        ranks = torch.zeros_like(S)
        rank_values = torch.arange(1, S.shape[1] + 1, device=device, dtype=torch.float).unsqueeze(0).expand_as(sorted_indices)
        ranks.scatter_(1, sorted_indices, rank_values)
        S_rrf += 1.0 / (k_rrf + ranks)
    res = evaluate_retrieval_gpu(S_rrf, targets_gpu)
    res["Temps (s)"] = time.time() - t0
    results["Fusion : Reciprocal Rank (RRF)"] = res

    # --- 7. MIRAGE (Grid Search) ---
    t0 = time.time()
    S_mirage = sum(w * (torch.tensor(query_vecs[n], device=device) @ torch.tensor(gallery_vecs[n], device=device).T) for n, w in best_weights.items() if w > 0)
    res = evaluate_retrieval_gpu(S_mirage, targets_gpu)
    res["Temps (s)"] = time.time() - t0
    results["MIRAGE (Grid Search)"] = res

    return results

def display_and_save_results(results_dict, task_name, chemin_csv, chemin_md):
    """Formate, affiche en gras, et sauvegarde les résultats (3 décimales)."""
    print("\n" + "="*60)
    print(f"RÉSULTATS TEST - ABLATION STUDY (Tâche {task_name})")
    print("="*60)

    df = pd.DataFrame.from_dict(results_dict, orient='index')
    # On ajoute la colonne Temps
    df = df[['R@1', 'R@5', 'R@10', 'mAP', 'NDCG', 'Temps (s)']]

    df_formatted = df.copy()
    for col in df_formatted.columns:
        if col == 'Temps (s)':
            min_val = df[col].min() # Pour le temps, le plus bas est le meilleur !
            df_formatted[col] = df[col].apply(lambda x: f"**{x:.3f}**" if x == min_val else f"{x:.3f}")
        else:
            max_val = df[col].max() 
            df_formatted[col] = df[col].apply(lambda x: f"**{x:.3f}**" if x == max_val else f"{x:.3f}")
        
    display(Markdown(df_formatted.to_markdown()))
    
    df.to_csv(chemin_csv, float_format="%.3f")
    df_formatted.to_markdown(chemin_md)

## Exécution de l'Évaluation

In [5]:
results_t2i = run_ablation_study(
    query_vecs=txt_vecs_test, 
    gallery_vecs=img_vecs_test, 
    model_names=model_names_t2i, 
    targets_gpu=targets_t2i_gpu, 
    best_weights=BEST_WEIGHTS['t2i'][METRIC_CIBLE], 
    device=device
)
display_and_save_results(results_t2i, task_name="T2I", chemin_csv=config.metriques_t2i_csv, chemin_md=config.metriques_t2i_md)

results_i2t = run_ablation_study(
    query_vecs=img_vecs_test, 
    gallery_vecs=txt_vecs_test, 
    model_names=model_names_i2t, 
    targets_gpu=targets_i2t_gpu, 
    best_weights=BEST_WEIGHTS['i2t'][METRIC_CIBLE], 
    device=device
)
display_and_save_results(results_i2t, task_name="I2T", chemin_csv=config.metriques_i2t_csv, chemin_md=config.metriques_i2t_md)


RÉSULTATS TEST - ABLATION STUDY (Tâche T2I)


|                                | R@1       | R@5       | R@10      | mAP       | NDCG      | Temps (s)   |
|:-------------------------------|:----------|:----------|:----------|:----------|:----------|:------------|
| siglip seul                    | 0.830     | 0.961     | 0.980     | 0.889     | 0.915     | 0.150       |
| blip seul                      | 0.821     | 0.961     | 0.978     | 0.883     | 0.911     | **0.044**   |
| convnext_clip seul             | 0.790     | 0.941     | 0.972     | 0.857     | 0.891     | 0.045       |
| eva_clip seul                  | 0.773     | 0.937     | 0.968     | 0.846     | 0.882     | 0.044       |
| Fusion : Addition Simple       | 0.863     | 0.972     | 0.987     | 0.912     | 0.933     | 0.049       |
| Fusion : Multiplication        | 0.860     | 0.972     | 0.985     | 0.910     | 0.931     | 0.056       |
| Fusion : Concaténation         | 0.863     | 0.972     | 0.987     | 0.912     | 0.933     | 0.082       |
| Fusion : Maximum               | 0.842     | 0.971     | 0.985     | 0.899     | 0.923     | 0.059       |
| Fusion : Reciprocal Rank (RRF) | 0.847     | 0.968     | 0.984     | 0.901     | 0.925     | 0.056       |
| MIRAGE (Grid Search)           | **0.868** | **0.974** | **0.988** | **0.916** | **0.936** | 0.049       |


RÉSULTATS TEST - ABLATION STUDY (Tâche I2T)


|                                | R@1       | R@5       | R@10      | mAP       | NDCG      | Temps (s)   |
|:-------------------------------|:----------|:----------|:----------|:----------|:----------|:------------|
| siglip seul                    | 0.942     | 0.997     | 0.998     | 0.856     | 0.934     | 0.013       |
| blip seul                      | 0.936     | 0.995     | 0.999     | 0.834     | 0.922     | **0.011**   |
| convnext_clip seul             | 0.917     | 0.992     | 0.997     | 0.815     | 0.911     | 0.013       |
| eva_clip seul                  | 0.897     | 0.985     | 0.992     | 0.797     | 0.900     | 0.012       |
| Fusion : Addition Simple       | **0.960** | **1.000** | **1.000** | 0.883     | **0.948** | 0.017       |
| Fusion : Multiplication        | 0.959     | 0.999     | **1.000** | 0.882     | 0.947     | 0.017       |
| Fusion : Concaténation         | **0.960** | **1.000** | **1.000** | 0.883     | 0.948     | 0.048       |
| Fusion : Maximum               | 0.947     | 0.996     | 0.999     | 0.863     | 0.937     | 0.018       |
| Fusion : Reciprocal Rank (RRF) | 0.959     | 0.998     | **1.000** | 0.874     | 0.943     | 0.023       |
| MIRAGE (Grid Search)           | **0.960** | 0.999     | **1.000** | **0.883** | 0.948     | 0.017       |